In [12]:
from datetime import datetime

TS_FMT = "%Y-%m-%d %H:%M:%S"


def detect_timestomping(file_meta, change_gap_minutes=60):
    """
    file_meta:
    {
        "modified": "...",
        "accessed": "...",
        "changed": "...",
        "born": "..."
    }

    Returns:
        (is_suspicious: bool, reasons: list)
    """

    m = datetime.strptime(file_meta["modified"], TS_FMT)
    a = datetime.strptime(file_meta["accessed"], TS_FMT)
    c = datetime.strptime(file_meta["changed"], TS_FMT)
    b = datetime.strptime(file_meta["born"], TS_FMT)

    reasons = []

    # Modified before creation
    if m < b:
        reasons.append(
            "Modified time is earlier than Born (creation) time"
        )

    # Accessed before creation
    if a < b:
        reasons.append(
            "Accessed time is earlier than Born (creation) time"
        )

    # Large gap between Changed and Modified timestamps
    gap_minutes = abs((c - m).total_seconds()) / 60

    if gap_minutes > change_gap_minutes and c > m:
        reasons.append(
            f"MFT Changed time is {gap_minutes:.0f} minutes after "
            "Modified time - metadata may have been altered after the fact"
        )

    return (len(reasons) > 0, reasons)

In [13]:
file_meta = {
    "modified": "2026-08-04 09:00:00",
    "accessed": "2026-08-04 09:10:00",
    "changed": "2026-08-04 12:30:00",
    "born": "2026-08-04 10:00:00"
}

result = detect_timestomping(file_meta)

print(result)

(True, ['Modified time is earlier than Born (creation) time', 'Accessed time is earlier than Born (creation) time', 'MFT Changed time is 210 minutes after Modified time - metadata may have been altered after the fact'])
